In [3]:
import pickle
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

# -----------------------------
# 1. Load data
# -----------------------------
data = pd.read_csv("cust_latest.csv")

# -----------------------------
# 2. Clean and prepare date
# -----------------------------
data["PO Date"] = pd.to_datetime(data["PO Date"])

data["day"] = data["PO Date"].dt.day
data["month"] = data["PO Date"].dt.month
data["year"] = data["PO Date"].dt.year
data["day_of_week"] = data["PO Date"].dt.dayofweek
# -----------------------------
# 3. New features
# -----------------------------
data = data.sort_values("PO Date")

data["prev_qty"] = data["Bill Qty"].shift(1)
data["rolling_avg_7"] = data["Bill Qty"].rolling(7).mean()
data["prev_qty_7"] = data["Bill Qty"].shift(7)
data["prev_qty_30"] = data["Bill Qty"].shift(30)

data["day_of_week"] = data["PO Date"].dt.dayofweek
data["quarter"] = data["PO Date"].dt.quarter

data = data.dropna()

# -----------------------------
# 4. Define X and y
# -----------------------------
target_col = "Bill Qty"

feature_cols = [
    "day",
    "month",
    "year",
    "day_of_week",
    "quarter",
    "prev_qty",
    "prev_qty_7",
    "prev_qty_30",
    "rolling_avg_7",
    "Brand Name",
    "Material Family"
]

X = data[feature_cols]
y = data[target_col]

# -----------------------------
# 5. Identify column types
# -----------------------------
numeric_features = [
    "day",
    "month",
    "year",
    "day_of_week",
    "quarter",
    "prev_qty",
    "prev_qty_7",
    "prev_qty_30",
    "rolling_avg_7"
]

categorical_features = [
    "Brand Name",
    "Material Family"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# -----------------------------
# 6. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

# -----------------------------
# 7. Create models
# -----------------------------
lin_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_split=5,
        random_state=42
    ))
])

svr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVR(C=10, epsilon=0.1, kernel="rbf"))
])

voting_model = VotingRegressor([
    ("lin", lin_model),
    ("rf", rf_model),
    ("svr", svr_model)
])


models = {
    "LinearRegression": lin_model,
    "RandomForest": rf_model,
    "SVR": svr_model,
    "VotingRegressor": voting_model
}

# -----------------------------
# 8. Train and evaluate
# -----------------------------
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    print(f"{name} MSE: {mse:.4f}")

# -----------------------------
# 9. Save model
# -----------------------------
with open("predicted_sales_model.pkl", "wb") as f:
    pickle.dump({
        "model": voting_model,
        "features": feature_cols,
        "brand_options": sorted(data["Brand Name"].dropna().unique()),
        "material_options": sorted(data["Material Family"].dropna().unique())
    }, f)

print("Saved predicted_sales_model.pkl")

LinearRegression MSE: 438.3145
RandomForest MSE: 319.8526
SVR MSE: 590.3054
VotingRegressor MSE: 361.8569
Saved predicted_sales_model.pkl
